# Camada Silver TS_ALUNO (Microdados INEP)

**Pipeline:** Análise da Alfabetização no Brasil
**Responsabilidade:** ler o TS_ALUNO da Bronze (Parquet), limpar e normalizar — sem cortar colunas.

**Escopo desta camada:** mantém as 27 colunas. A seleção do que interessa para análise (proficiência, alfabetização) acontece na Gold, não aqui — a Silver só garante qualidade do dado, não decide o que será usado.

## 1. Configuração de Acesso

Mesmo padrão dos demais notebooks — conexão ao ADLS Gen2 via Azure Key Vault.

In [1]:
# ============================================================
# Configuração de acesso ao ADLS Gen2 via Azure Key Vault
# ============================================================
from notebookutils import mssparkutils

CONTA_STORAGE = "stalfalfabetizacao"

CHAVE_ACESSO = mssparkutils.credentials.getSecret(
    "https://kv-alfabetizacao.vault.azure.net/",
    "storage-account-key"
)

spark.conf.set(
    f"fs.azure.account.key.{CONTA_STORAGE}.dfs.core.windows.net",
    CHAVE_ACESSO
)

CAMINHO_BRONZE = f"abfss://bronze@{CONTA_STORAGE}.dfs.core.windows.net/"
CAMINHO_SILVER = f"abfss://silver@{CONTA_STORAGE}.dfs.core.windows.net/"

print("Conexão configurada com sucesso!")
print(f"  Bronze: {CAMINHO_BRONZE}")
print(f"  Silver: {CAMINHO_SILVER}")

StatementMeta(sparkpool, 13, 2, Finished, Available, Finished, False)

Conexão configurada com sucesso!
  Bronze: abfss://bronze@stalfalfabetizacao.dfs.core.windows.net/
  Silver: abfss://silver@stalfalfabetizacao.dfs.core.windows.net/


## 2. Leitura e Transformação

Leitura do TS_ALUNO em Parquet. Limpeza aplicada: remoção de duplicados, tratamento de nulos nas chaves e correção do registro de alunos ausentes na prova.

In [2]:
# ============================================================
# Leitura da Bronze e limpeza do TS_ALUNO
# Regras aplicadas:
#   - Remoção de duplicados
#   - Nulos apenas nas colunas-chave (ID_ALUNO, CO_MUNICIPIO)
#   - Alunos ausentes (IN_PRESENCA_LP = 0) são mantidos como histórico,
#     mas ficam sinalizados — a Gold decide se filtra ou não
# ============================================================
from pyspark.sql.functions import col, trim, upper

df_ts_aluno = spark.read.parquet(CAMINHO_BRONZE + "ts_aluno/")

df_ts_aluno_silver = df_ts_aluno \
    .dropDuplicates() \
    .dropna(subset=["ID_ALUNO", "CO_MUNICIPIO"]) \
    .withColumn("SG_UF", upper(trim(col("SG_UF"))))

print(f"TS_ALUNO Silver: {df_ts_aluno_silver.count()} linhas")
print(f"  Presentes na prova:  {df_ts_aluno_silver.filter(col('IN_PRESENCA_LP') == 1).count()}")
print(f"  Ausentes na prova:   {df_ts_aluno_silver.filter(col('IN_PRESENCA_LP') == 0).count()}")

StatementMeta(sparkpool, 13, 3, Finished, Available, Finished, False)

TS_ALUNO Silver: 2222164 linhas
  Presentes na prova:  1969411
  Ausentes na prova:   252753


## 3. Gravação em Parquet

Persistência do TS_ALUNO tratado na Silver.

In [3]:
# ============================================================
# Gravação do TS_ALUNO Silver em Parquet
# ============================================================

df_ts_aluno_silver.write.mode("overwrite").parquet(CAMINHO_SILVER + "ts_aluno/")

print("TS_ALUNO Silver gravado com sucesso!")
print(f"  {CAMINHO_SILVER}ts_aluno/")

StatementMeta(sparkpool, 13, 4, Finished, Available, Finished, False)

TS_ALUNO Silver gravado com sucesso!
  abfss://silver@stalfalfabetizacao.dfs.core.windows.net/ts_aluno/
